In [1]:
# ============================================================
# INGEST SPRINTS — LOCAL EXECUTION
# ============================================================

import os
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DateType
)
import nbformat
from nbconvert import PythonExporter

In [ ]:
# -----------------------------------------
# 1. Load environment + helpers
# -----------------------------------------
def run_notebook(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    source, _ = PythonExporter().from_notebook_node(nb)
    exec(source, globals())

run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")
run_notebook("/Users/manoharazalki/F1-Analytics/notebooks/00-common/02_bronze_helpers.ipynb")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 13:23:56 WARN Utils: Your hostname, Manohars-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.68.78.178 instead (on interface en0)
26/06/09 13:23:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 13:23:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 13:23:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 13:23:56 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/09 13:23:56 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/09 13:23:56 

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [3]:
# -----------------------------------------
# 2. Detect latest landing batch folder
# -----------------------------------------
batch_folders = sorted([
    f for f in os.listdir(LANDING_PATH)
    if os.path.isdir(os.path.join(LANDING_PATH, f))
])

if not batch_folders:
    raise Exception("❌ No batch folders found in landing!")

p_batch_id = batch_folders[-1]
print("Detected landing batch folder:", p_batch_id)

BATCH_LANDING_PATH = f"{LANDING_PATH}/{p_batch_id}"

Detected landing batch folder: 2025-01


In [4]:
# -----------------------------------------
# 3. Generate pipeline batch_id
# -----------------------------------------
batch_id = generate_batch_id()
print("Pipeline batch_id:", batch_id)

Pipeline batch_id: 20260609_132357


In [5]:
# -----------------------------------------
# 4. Define source + target paths
# -----------------------------------------
source_folder = f"{BATCH_LANDING_PATH}/sprints"
target_path = f"{BRONZE_PATH}/sprints"
os.makedirs(target_path, exist_ok=True)

print("Source folder:", source_folder)
print("Target path:", target_path)

Source folder: /Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints
Target path: /Users/manoharazalki/F1-Analytics/data/bronze/sprints


In [6]:
# -----------------------------------------
# 5. Convert JSON arrays → JSON lines
# -----------------------------------------
clean_files = []

for file in os.listdir(source_folder):
    if not file.endswith(".json"):
        continue

    input_path = f"{source_folder}/{file}"
    output_path = f"{source_folder}/clean_{file}"

    try:
        with open(input_path, "r") as f:
            data = json.load(f)

        # If file is a JSON array → convert to JSON lines
        if isinstance(data, list):
            with open(output_path, "w") as out:
                for row in data:
                    out.write(json.dumps(row) + "\n")
            clean_files.append(output_path)
            print(f"✔ Converted array JSON → JSON lines: {file}")

        # Already JSON lines
        else:
            clean_files.append(input_path)
            print(f"✔ JSON lines file detected: {file}")

    except Exception as e:
        print(f"Skipping invalid JSON file: {file}")
        print("Error:", e)

print("Clean files to load:", clean_files)

✔ Converted array JSON → JSON lines: sprints_2021.json
✔ Converted array JSON → JSON lines: sprints_2024.json
✔ Converted array JSON → JSON lines: sprints_2025.json
✔ Converted array JSON → JSON lines: sprints_2022.json
✔ Converted array JSON → JSON lines: sprints_2023.json
Clean files to load: ['/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2021.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2024.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2025.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2022.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2023.json']


In [7]:
# -----------------------------------------
# 6. Validate JSON files
# -----------------------------------------
valid_files = []
invalid_files = []

for file in clean_files:
    try:
        with open(file, "r") as f:
            for line in f:
                json.loads(line)
        valid_files.append(file)
    except Exception as e:
        print(f"Invalid JSON file skipped: {file}")
        print("   Error:", e)
        invalid_files.append(file)

print("Valid JSON files:", valid_files)
print("Invalid JSON files:", invalid_files)

if not valid_files:
    raise Exception("❌ No valid JSON files to load!")

Valid JSON files: ['/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2021.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2024.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2025.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2022.json', '/Users/manoharazalki/F1-Analytics/data/landing/2025-01/sprints/clean_sprints_2023.json']
Invalid JSON files: []


In [8]:
# -----------------------------------------
# 7. Define schema
# -----------------------------------------
sprint_schema = StructType([
    StructField("date", DateType(), True),
    StructField("raceName", StringType(), True),
    StructField("round", IntegerType(), True),
    StructField("season", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("constructorId", StringType(), True),
    StructField("driverId", StringType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("number", IntegerType(), True),
    StructField("points", FloatType(), True),
    StructField("position", IntegerType(), True),
    StructField("positionText", StringType(), True),
    StructField("status", StringType(), True)
])

In [9]:
# -----------------------------------------
# 8. Read cleaned JSON files
# -----------------------------------------
sprints_df = (
    spark.read
        .format("json")
        .schema(sprint_schema)
        .option("mode", "FAILFAST")
        .load(valid_files)
)

print("✔ Sprints JSON loaded successfully")
sprints_df.show(5, truncate=False)

✔ Sprints JSON loaded successfully
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |11    |8.0   |1       |1           |Finished|
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|ferrari      |leclerc       |1   |17  |16    |7.0   |2       |2           |Finished|
|2023-04-30|azerbaij

In [10]:
# -----------------------------------------
# 9. Add ingestion metadata
# -----------------------------------------
sprints_final_df = add_ingestion_metadata(sprints_df, source_folder)

print("✔ Metadata added")
sprints_final_df.show(5, truncate=False)

✔ Metadata added
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+-------------------------+--------------------------------------------------------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp         |source_file                                                   |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+-------------------------+--------------------------------------------------------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull     |perez         |2   |17  |11    

In [11]:
# -----------------------------------------
# 10. Write to Bronze (partitioned by batch_id)
# -----------------------------------------
write_to_bronze(
    sprints_final_df,
    f"{target_path}/data.parquet",
    batch_id
)

print(f"✔ Sprints Bronze written to: {target_path}/data.parquet")

✔ Sprints Bronze written to: /Users/manoharazalki/F1-Analytics/data/bronze/sprints/data.parquet


In [12]:
# -----------------------------------------
# 11. Read back for validation
# -----------------------------------------
spark.read.parquet(f"{target_path}/data.parquet").show(5, truncate=False)

+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|date      |raceName             |round|season|url                                                     |constructorId|driverId      |grid|laps|number|points|position|positionText|status  |ingest_timestamp          |source_file                                                   |batch_id       |
+----------+---------------------+-----+------+--------------------------------------------------------+-------------+--------------+----+----+------+------+--------+------------+--------+--------------------------+--------------------------------------------------------------+---------------+
|2023-04-30|azerbaijan grand prix|4    |2023  |https://en.wikipedia.org/wiki/2023_Azerbaijan_Grand_Prix|red_bull   